# Credit Scoring MLOps

## Preprocessing & Feature Engineering

L'objectif de ce notebook est de :
- nettoyer la table principale `application_train/test` (anomalies, NaN) ;
- créer des **features pré-imputation** (flags binaires issus des anomalies) ;
- agréger les tables secondaires par `SK_ID_CURR` ;
- fusionner toutes les tables en un seul DataFrame ;
- imputer les valeurs manquantes (médiane/mode calculés sur le train) ;
- construire les **features principales** (ratios, âge…) sur le dataset complet et imputé ;
- encoder les variables catégorielles ;
- sauvegarder les données prêtes pour la modélisation.

Les décisions prises ici sont basées sur les observations du notebook `1_eda.ipynb`.

## 0. Setup & imports

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import src.utils as utils

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', palette='muted')

DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'

---
## 1. Nettoyage de `application_train/test`

### 1.1 Chargement

In [2]:
app_train = pd.read_csv(DATA_RAW + 'application_train.csv')
app_test  = pd.read_csv(DATA_RAW + 'application_test.csv')

print(f'Train : {app_train.shape} | Test : {app_test.shape}')

Train : (307511, 122) | Test : (48744, 121)


### 1.2 Suppression des colonnes avec trop de NaN
- Seuil : >50% de valeurs manquantes
- Exception : `EXT_SOURCE_1` (~56% NaN mais très prédictif → à conserver)

In [3]:
# Colonnes à conserver malgré un taux NaN élevé (signal fort en EDA)
exceptions = ['EXT_SOURCE_1']

# Calcul du taux de NaN par colonne sur le train
nan_ratio = app_train.isnull().mean()
cols_a_supprimer = nan_ratio[nan_ratio > 0.5].index.tolist()
cols_a_supprimer = [c for c in cols_a_supprimer if c not in exceptions]

print(f'{len(cols_a_supprimer)} colonnes supprimées (>50% NaN)')

app_train = app_train.drop(columns=cols_a_supprimer)
app_test  = app_test.drop(columns=[c for c in cols_a_supprimer if c in app_test.columns])

print(f'Train : {app_train.shape} | Test : {app_test.shape}')

40 colonnes supprimées (>50% NaN)
Train : (307511, 82) | Test : (48744, 81)


### 1.3 Correction des anomalies
- `DAYS_EMPLOYED = 365243` → remplacer par `NaN` + flag binaire `DAYS_EMPLOYED_ANOM`
- `CODE_GENDER = 'XNA'` (4 individus) → remplacer par `'F'` (mode)

In [4]:
# DAYS_EMPLOYED : valeur aberrante 365243 → remplacer par NaN
for df in [app_train, app_test]:
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# CODE_GENDER : 'XNA' (4 individus) → remplacer par le mode ('F')
for df in [app_train, app_test]:
    df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', 'F')

print('Anomalies corrigées')

Anomalies corrigées


### 1.4 Features pré-imputation — flags d'anomalies
- `DAYS_EMPLOYED_ANOM` : flag binaire indiquant la valeur aberrante 365243 (doit être créé **avant** l'imputation pour ne pas perdre l'information)

In [5]:
# Flag binaire : 1 si DAYS_EMPLOYED était aberrant (365243), 0 sinon
# Créé avant l'imputation pour ne pas perdre l'information
for df in [app_train, app_test]:
    df['DAYS_EMPLOYED_ANOM'] = df['DAYS_EMPLOYED'].isnull().astype(int)

print('Flag DAYS_EMPLOYED_ANOM créé')
print(app_train['DAYS_EMPLOYED_ANOM'].value_counts())

Flag DAYS_EMPLOYED_ANOM créé
DAYS_EMPLOYED_ANOM
0    252137
1     55374
Name: count, dtype: int64


---
## 2. Agrégation des tables secondaires

### 2.1 `bureau.csv`
Features globales : `bureau_count`, `bureau_debt_mean`, `bureau_overdue_mean`, `bureau_active_count`

Features par statut (inspiration kernel Kaggle) :
- Crédits **actifs** : `actif_debt_mean`, `actif_count`
- Crédits **clôturés** : `cloture_debt_mean`, `cloture_count`

In [6]:
bureau = pd.read_csv(DATA_RAW + 'bureau.csv')

# Agrégation globale
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_count        = ('SK_ID_BUREAU', 'count'),
    bureau_debt_mean    = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    bureau_overdue_mean = ('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    bureau_active_count = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
).reset_index()

# Split crédits actifs vs clôturés (signaux différents selon le statut)
actif   = bureau[bureau['CREDIT_ACTIVE'] == 'Active']
cloture = bureau[bureau['CREDIT_ACTIVE'] == 'Closed']

actif_agg = actif.groupby('SK_ID_CURR').agg(
    actif_debt_mean = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    actif_count     = ('SK_ID_BUREAU', 'count'),
).reset_index()

cloture_agg = cloture.groupby('SK_ID_CURR').agg(
    cloture_debt_mean = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    cloture_count     = ('SK_ID_BUREAU', 'count'),
).reset_index()

bureau_agg = bureau_agg.merge(actif_agg,   on='SK_ID_CURR', how='left')
bureau_agg = bureau_agg.merge(cloture_agg, on='SK_ID_CURR', how='left')

print(f'bureau_agg : {bureau_agg.shape}')
bureau_agg.head(3)

bureau_agg : (305811, 9)


,SK_ID_CURR,bureau_count,bureau_debt_mean,bureau_overdue_mean,bureau_active_count,actif_debt_mean,actif_count,cloture_debt_mean,cloture_count
0,100001,7,85240.928571,0.0,3,198895.5,3.0,0.0,4.0
1,100002,8,49156.200000,0.0,2,122890.5,2.0,0.0,6.0
2,100003,4,0.000000,0.0,1,0.0,1.0,0.0,3.0


### 2.2 `previous_application.csv`
Features globales : `prev_count`, `prev_refused_count`, `taux_refus`, `prev_credit_mean`

Features par statut (inspiration kernel Kaggle) :
- Demandes **approuvées** : `approve_credit_mean`, `approve_count`
- Demandes **refusées** : `refuse_credit_mean`, `refuse_count`

In [7]:
prev = pd.read_csv(DATA_RAW + 'previous_application.csv')

# Agrégation globale
prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_count         = ('SK_ID_PREV', 'count'),
    prev_refused_count = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_credit_mean   = ('AMT_CREDIT', 'mean'),
).reset_index()

prev_agg['taux_refus'] = prev_agg['prev_refused_count'] / prev_agg['prev_count']

# Split demandes approuvées vs refusées (comportements différents selon le statut)
approve = prev[prev['NAME_CONTRACT_STATUS'] == 'Approved']
refuse  = prev[prev['NAME_CONTRACT_STATUS'] == 'Refused']

approve_agg = approve.groupby('SK_ID_CURR').agg(
    approve_credit_mean = ('AMT_CREDIT', 'mean'),
    approve_count       = ('SK_ID_PREV', 'count'),
).reset_index()

refuse_agg = refuse.groupby('SK_ID_CURR').agg(
    refuse_credit_mean = ('AMT_CREDIT', 'mean'),
    refuse_count       = ('SK_ID_PREV', 'count'),
).reset_index()

prev_agg = prev_agg.merge(approve_agg, on='SK_ID_CURR', how='left')
prev_agg = prev_agg.merge(refuse_agg,  on='SK_ID_CURR', how='left')

print(f'prev_agg : {prev_agg.shape}')
prev_agg.head(3)

prev_agg : (338857, 9)


,SK_ID_CURR,prev_count,prev_refused_count,prev_credit_mean,taux_refus,approve_credit_mean,approve_count,refuse_credit_mean,refuse_count
0,100001,1,0,23787.0,0.0,23787.0,1.0,NaN,NaN
1,100002,1,0,179055.0,0.0,179055.0,1.0,NaN,NaN
2,100003,3,0,484191.0,0.0,484191.0,3.0,NaN,NaN


### 2.3 `POS_CASH_balance.csv`
Features à construire : `pos_dpd_mean`, `pos_dpd_max`

In [8]:
pos = pd.read_csv(DATA_RAW + 'POS_CASH_balance.csv')

pos_agg = pos.groupby('SK_ID_CURR').agg(
    pos_dpd_mean = ('SK_DPD', 'mean'),
    pos_dpd_max  = ('SK_DPD', 'max'),
).reset_index()

print(f'pos_agg : {pos_agg.shape}')
pos_agg.head(3)

pos_agg : (337252, 3)


,SK_ID_CURR,pos_dpd_mean,pos_dpd_max
0,100001,0.777778,7
1,100002,0.000000,0
2,100003,0.000000,0


### 2.4 `installments_payments.csv`
Features à construire : `inst_retard_mean`, `inst_retard_max`, `inst_diff_mean`, `a_eu_retard`

In [9]:
inst = pd.read_csv(DATA_RAW + 'installments_payments.csv')

# Retard en jours (positif = en retard)
inst['retard_jours'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
# Différence de montant payé vs dû
inst['diff_montant'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_retard_mean = ('retard_jours', 'mean'),
    inst_retard_max  = ('retard_jours', 'max'),
    inst_diff_mean   = ('diff_montant', 'mean'),
    a_eu_retard      = ('retard_jours', lambda x: (x > 0).any().astype(int)),
).reset_index()

print(f'inst_agg : {inst_agg.shape}')
inst_agg.head(3)

inst_agg : (339587, 5)


,SK_ID_CURR,inst_retard_mean,inst_retard_max,inst_diff_mean,a_eu_retard
0,100001,-7.285714,11.0,0.0,1
1,100002,-20.421053,-12.0,0.0,0
2,100003,-7.160000,-1.0,0.0,0


### 2.5 `credit_card_balance.csv`
Features à construire : `cc_utilisation_mean`, `cc_dpd_mean`, `cc_balance_mean`

In [10]:
cc = pd.read_csv(DATA_RAW + 'credit_card_balance.csv')

# Taux d'utilisation : balance / limite de crédit
cc['utilisation'] = cc['AMT_BALANCE'] / cc['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)

cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_utilisation_mean = ('utilisation', 'mean'),
    cc_dpd_mean         = ('SK_DPD', 'mean'),
    cc_balance_mean     = ('AMT_BALANCE', 'mean'),
).reset_index()

print(f'cc_agg : {cc_agg.shape}')
cc_agg.head(3)

cc_agg : (103558, 4)


,SK_ID_CURR,cc_utilisation_mean,cc_dpd_mean,cc_balance_mean
0,100006,0.000000,0.000000,0.000000
1,100011,0.302678,0.000000,54482.111149
2,100013,0.115301,0.010417,18159.919219


---
## 3. Fusion de toutes les tables

- `left join` depuis `app_train` / `app_test` sur `SK_ID_CURR`
- Les clients sans table secondaire auront des `NaN` → gérés à l'imputation

In [11]:
tables_secondaires = [bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg]

train = app_train.copy()
test  = app_test.copy()

for table in tables_secondaires:
    train = train.merge(table, on='SK_ID_CURR', how='left')
    test  = test.merge(table, on='SK_ID_CURR', how='left')

print(f'Train fusionné : {train.shape}')
print(f'Test  fusionné : {test.shape}')

Train fusionné : (307511, 108)
Test  fusionné : (48744, 107)


---
## 4. Imputation des valeurs manquantes

- Variables numériques → médiane
- Variables catégorielles → mode
- `EXT_SOURCE_1/2/3` → médiane (signal fort, à conserver malgré les NaN)

> Attention : calculer les statistiques d'imputation **sur le train uniquement**, puis appliquer sur le test.

In [12]:
# Calcul des statistiques d'imputation sur le train uniquement
num_cols = train.select_dtypes(include='number').columns.tolist()
cat_cols = train.select_dtypes(include='object').columns.tolist()

# Exclure la cible du calcul
num_cols = [c for c in num_cols if c != 'TARGET']

medianes = train[num_cols].median()
modes    = train[cat_cols].mode().iloc[0]

# Application sur train et test
train[num_cols] = train[num_cols].fillna(medianes)
train[cat_cols] = train[cat_cols].fillna(modes)

test[num_cols] = test[num_cols].fillna(medianes)
test[cat_cols] = test[cat_cols].fillna(modes)

# Vérification
print(f'NaN restants — train : {train.isnull().sum().sum()} | test : {test.isnull().sum().sum()}')

/tmp/ipykernel_2058/1689548014.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include='object').columns.tolist()


NaN restants — train : 0 | test : 0


---
## 5. Feature engineering principal

Dataset complet et imputé — plus aucun NaN, on peut calculer les ratios sans risque.

Features issues de l'EDA (`1_eda.ipynb` section 1.7) et du kernel Kaggle de référence :

| Feature | Formule | Corrélation TARGET | Source |
|---|---|---|---|
| `AGE_YEARS` | `-DAYS_BIRTH / 365` | -0.078 | EDA |
| `DAYS_EMPLOYED_PERC` | `DAYS_EMPLOYED / DAYS_BIRTH` | -0.068 | Kernel |
| `RATIO_ANNUITE_REVENU` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | +0.014 | EDA + Kernel |
| `PAYMENT_RATE` | `AMT_ANNUITY / AMT_CREDIT` | +0.013 | Kernel |
| `RATIO_CREDIT_REVENU` | `AMT_CREDIT / AMT_INCOME_TOTAL` | ~0.002 | EDA |

> `RATIO_CREDIT_ANNUITE` supprimé : redondant avec `PAYMENT_RATE` (formule inverse).

In [13]:
for df in [train, test]:
    # Âge en années (DAYS_BIRTH toujours négatif, toujours non nul)
    df['AGE_YEARS']            = -df['DAYS_BIRTH'] / 365
    # Durée d'emploi relative à l'âge (signal le plus fort parmi les ratios : -0.068)
    df['DAYS_EMPLOYED_PERC']   = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    # Part du revenu consacrée aux annuités
    df['RATIO_ANNUITE_REVENU'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    # Taux de remboursement mensuel
    df['PAYMENT_RATE']         = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    # Ratio crédit / revenu (signal faible mais conservé)
    df['RATIO_CREDIT_REVENU']  = df['AMT_CREDIT']  / df['AMT_INCOME_TOTAL']

nouvelles_features = ['AGE_YEARS', 'DAYS_EMPLOYED_PERC', 'RATIO_ANNUITE_REVENU',
                      'PAYMENT_RATE', 'RATIO_CREDIT_REVENU']
print('Features créées :')
train[nouvelles_features].describe().round(3)

Features créées :


,AGE_YEARS,DAYS_EMPLOYED_PERC,RATIO_ANNUITE_REVENU,PAYMENT_RATE,RATIO_CREDIT_REVENU
count,307511.000,307511.000,307511.000,307511.000,307511.000
mean,43.937,0.142,0.181,0.054,3.958
std,11.956,0.125,0.095,0.022,2.690
min,20.518,-0.000,0.000,0.017,0.005
25%,34.008,0.067,0.115,0.037,2.019
50%,43.151,0.091,0.163,0.050,3.265
75%,53.923,0.191,0.229,0.064,5.160
max,69.121,0.729,1.876,0.158,84.737


---
## 6. Encodage des variables catégorielles

- Regroupement des modalités rares (<1% des effectifs) en `'Autre'` avant encodage
- One-hot encoding pour les variables à faible cardinalité

In [14]:
cat_cols = train.select_dtypes(include='object').columns.tolist()

# Regroupement des modalités rares (<1%) en 'Autre'
for col in cat_cols:
    frequences = train[col].value_counts(normalize=True)
    modalites_rares = frequences[frequences < 0.01].index
    train[col] = train[col].replace(modalites_rares, 'Autre')
    test[col]  = test[col].replace(modalites_rares, 'Autre')

# One-hot encoding — drop_first=True pour éviter la multicolinéarité
train = pd.get_dummies(train, columns=cat_cols, drop_first=True)
test  = pd.get_dummies(test,  columns=cat_cols, drop_first=True)

# Alignement des colonnes train/test (le test peut manquer certaines modalités)
train, test = train.align(test, join='left', axis=1, fill_value=0)

# La colonne TARGET est dans train uniquement → la remettre à sa vraie valeur
# (align() l'a remplie à 0 dans test, ce qui est correct car test n'a pas de TARGET)

print(f'Train : {train.shape} | Test : {test.shape}')

/tmp/ipykernel_2058/3264041147.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include='object').columns.tolist()


Train : (307511, 160) | Test : (48744, 160)


---
## 7. Vérification finale

- Plus aucun NaN
- Shapes cohérentes train / test
- Colonnes identiques (hors `TARGET`)

In [15]:
nan_train = train.isnull().sum().sum()
nan_test  = test.isnull().sum().sum()

# Colonnes communes (hors TARGET)
cols_train = set(train.columns) - {'TARGET'}
cols_test  = set(test.columns)  - {'TARGET'}
diff = cols_train.symmetric_difference(cols_test)

print(f'NaN train : {nan_train} | NaN test : {nan_test}')
print(f'Shape train : {train.shape} | Shape test : {test.shape}')
print(f'Colonnes différentes (hors TARGET) : {len(diff)} → {diff if diff else "aucune"}')
assert nan_train == 0 and nan_test == 0, 'Il reste des NaN !'
assert len(diff) == 0, f'Colonnes non alignées : {diff}'
print('Vérification OK')

NaN train : 0 | NaN test : 0
Shape train : (307511, 160) | Shape test : (48744, 160)
Colonnes différentes (hors TARGET) : 0 → aucune
Vérification OK


---
## 8. Sauvegarde

Export dans `data/processed/` pour le notebook de modélisation.

In [16]:
import os
os.makedirs(DATA_PROCESSED, exist_ok=True)

train.to_csv(DATA_PROCESSED + 'train_processed.csv', index=False)
test.to_csv(DATA_PROCESSED  + 'test_processed.csv',  index=False)

print(f'Sauvegardé dans {DATA_PROCESSED}')
print(f'  train_processed.csv : {train.shape}')
print(f'  test_processed.csv  : {test.shape}')

Sauvegardé dans ../data/processed/
  train_processed.csv : (307511, 160)
  test_processed.csv  : (48744, 160)
